# Conversational Recommendation Demo

## Objective
Assemble the project so far into one explicit end-to-end demonstration:

`user request -> parsed request -> retrieved candidates -> reranked finalists -> grounding evidence -> final response object`

## Inputs
- retrieval artifacts from notebook 02
- curated scenarios from `../data/curated/scenario_suite.json`
- shared pipeline helpers in `functions/`
- raw `items.parquet` and `reviews.parquet` accessed lazily through DuckDB

## Outputs
- a final deterministic response schema
- multiple end-to-end scenario walkthroughs
- intermediate states for inspection at every pipeline stage

## What The Full Pipeline Demonstrates
The full system is explicit. The parser interprets the request, retrieval proposes plausible candidates, reranking applies transparent policy, grounding attaches evidence, and the final response object turns those structured states into a user-facing recommendation without leaving the ranking decision opaque.


## Pipeline Recap

Each stage contributes something different:

1. **Parsing** turns messy user language into a compact contract.
2. **Retrieval** brings back a plausible neighborhood of items.
3. **Reranking** applies explicit constraints and preference logic.
4. **Grounding** attaches metadata and review evidence.
5. **Response generation** packages all of that into a readable final object.

None of these stages is supposed to do the others' job silently.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.grounding import assemble_grounding_evidence
from functions.io import (
    connect_catalog_views,
    load_cross_encoder,
    load_curated_scenarios,
    load_retrieval_artifacts,
    materialize_scenarios,
)
from functions.parsing import build_parser_catalog, parse_user_request
from functions.response import FINAL_RESPONSE_TEMPLATE, build_final_response
from functions.reranking import rerank_candidates
from functions.retrieval import retrieve_candidates_for_request

artifacts = load_retrieval_artifacts(load_encoder_model=True)
candidate_pool = artifacts["candidate_pool"]
parser_catalog = build_parser_catalog(candidate_pool)
scenario_suite = load_curated_scenarios()
demo_scenarios = materialize_scenarios(scenario_suite["demo_scenarios"], candidate_pool)
con = connect_catalog_views(include_reviews=True)
cross_encoder = load_cross_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Shared Interfaces

The demo relies on shared helpers under `functions/`, which means the notebooks are no longer separate islands. The same parsing, retrieval, reranking, and grounding logic used here is the same logic already exercised in notebooks 03 to 05.


In [2]:
interface_summary = pd.DataFrame(
    [
        {
            "stage": "parser",
            "input": "raw user query (+ optional reference context)",
            "output": "parsed request contract",
        },
        {"stage": "retrieval", "input": "parsed request", "output": "baseline candidate set"},
        {
            "stage": "reranking",
            "input": "parsed request + baseline candidates",
            "output": "reranked finalists",
        },
        {"stage": "grounding", "input": "parsed request + finalists", "output": "supporting evidence rows"},
        {
            "stage": "response",
            "input": "parsed request + finalists + evidence",
            "output": "final deterministic response object",
        },
    ]
)
response_schema = pd.DataFrame(
    [{"field": key, "type": type(value).__name__} for key, value in FINAL_RESPONSE_TEMPLATE.items()]
)

display(interface_summary)
display(response_schema)

,stage,input,output
0,parser,raw user query (+ optional reference context),parsed request contract
1,retrieval,parsed request,baseline candidate set
2,reranking,parsed request + baseline candidates,reranked finalists
3,grounding,parsed request + finalists,supporting evidence rows
4,response,parsed request + finalists + evidence,final deterministic response object


,field,type
0,original_query,NoneType
1,parsed_request,NoneType
2,retrieved_candidates,list
3,reranked_finalists,list
4,supporting_evidence,list
5,explanation_text,str
6,tradeoff_notes,list
7,clarification,dict


## End-To-End Assembly

The next cell runs the whole pipeline over the curated scenario suite. It keeps every intermediate state rather than jumping directly to the final answer, because this notebook is meant to be read as a walkthrough rather than a black-box demo.


In [3]:
pipeline_runs = []
for scenario in demo_scenarios:
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    retrieved = retrieve_candidates_for_request(parsed, artifacts, top_k=8)
    rerank_result = rerank_candidates(parsed, artifacts, top_k_retrieval=12, top_k_final=3)
    evidence = assemble_grounding_evidence(
        parsed,
        rerank_result["reranked_candidates"],
        con,
        candidate_pool,
        cross_encoder,
        top_review_evidence_per_item=1,
    )
    final_response = build_final_response(
        parsed,
        retrieved,
        rerank_result["reranked_candidates"],
        evidence,
    )
    pipeline_runs.append(
        {
            "scenario": scenario,
            "parsed_request": parsed,
            "retrieved_candidates": retrieved,
            "rerank_result": rerank_result,
            "evidence": evidence,
            "final_response": final_response,
        }
    )

run_summary = pd.DataFrame(
    [
        {
            "label": run["scenario"]["label"],
            "clarification_needed": run["parsed_request"]["clarification_needed"],
            "retrieved_count": len(run["retrieved_candidates"]),
            "finalist_count": len(run["rerank_result"]["reranked_candidates"]),
            "evidence_rows": len(run["evidence"]),
        }
        for run in pipeline_runs
    ]
)

display(run_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label,clarification_needed,retrieved_count,finalist_count,evidence_rows
0,budget_headphones_avoid_beats,False,8,3,6
1,reference_cheaper_headphones,False,8,3,6
2,induction_frying_pan,False,8,3,6
3,dumbbells_home_gym,False,8,3,6
4,lightweight_programming_laptop,False,8,3,6
5,gift_ambiguity,True,8,3,6


## Scenario Walkthroughs

For each request below, the notebook shows the parsed structure, the baseline retrieval output, the reranked finalists, the supporting evidence, and the final response object. That preserves the full system trace instead of hiding the recommendation behind fluent prose.


In [4]:
for run in pipeline_runs:
    print("=" * 100)
    print(run["scenario"]["title"])
    print(run["scenario"]["query"])

    print("Parsed request")
    print(json.dumps(run["parsed_request"], indent=2, ensure_ascii=False))

    print("Retrieved candidates")
    display(
        run["retrieved_candidates"][
            ["parent_asin", "source", "title", "price", "average_rating", "retrieval_score"]
        ].head(5)
    )

    print("Reranked finalists")
    display(
        run["rerank_result"]["reranked_candidates"][
            [
                "reranked_rank",
                "baseline_rank",
                "rank_shift",
                "parent_asin",
                "title",
                "price",
                "average_rating",
                "final_score",
            ]
        ]
    )

    print("Supporting evidence")
    display(
        run["evidence"][
            [
                "parent_asin",
                "evidence_source",
                "matched_topic",
                "matched_terms",
                "relevance_score",
                "evidence_text",
            ]
        ]
    )

    print("Final response object")
    print(json.dumps(run["final_response"], indent=2, ensure_ascii=False))

Highly rated headphones under budget
Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.
Parsed request
{
  "original_query": "Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",
  "user_intent": "constrained_search",
  "domain_hint": "electronics",
  "hard_constraints": {
    "max_price": 80.0,
    "min_rating": null,
    "must_include_terms": [
      "headphones"
    ],
    "use_case": null,
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [
    "highly_rated"
  ],
  "reference_items": [],
  "excluded_brands": [
    "Beats"
  ],
  "excluded_terms": [],
  "grounding_needs": [
    "price",
    "rating",
    "brand_constraint"
  ],
  "clarification_needed": false,
  "clarification_questions": []
}
Retrieved candidates


,parent_asin,source,title,price,average_rating,retrieval_score
0,B081RBT9HM,electronics,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",19.99,4.3,0.122747
1,B00C4FBQT8,electronics,Sony 900MHz Wireless Stereo Noise Reduction Headphones,119.95,3.9,0.106422
2,B0BZ6QTH4S,electronics,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",19.99,4.4,0.106393
3,B07DHZMQZS,electronics,Jabra Elite 65t Alexa Enabled True Wireless Earbuds with Charging Case IP55 rated - Titanium Black (Renewed),58.97,4.0,0.106393
4,B07NT6BMTK,electronics,"Buyer's Point Ultra High Speed HDMI 2.1 Cable CL3 Rated Dynamic HDR 1.8M(6ft) 8K 120Hz, 48Gbps, eARC (1 Pack, Black CL3 Rated)",11.99,4.6,0.106340


Reranked finalists


,reranked_rank,baseline_rank,rank_shift,parent_asin,title,price,average_rating,final_score
0,1,1,0,B081RBT9HM,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",19.99,4.3,0.868940
1,2,3,1,B0BZ6QTH4S,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",19.99,4.4,0.595355
2,3,12,9,B09YTGFYZK,"SAMSUNG Galaxy Earbuds Charging Case Cover Compatible with Buds2 Pro, Buds Pro, Buds Live, Buds2, IP67 Rated Water and Dust Resistant Protection, Clear",9.95,4.4,0.550650


Supporting evidence


...

Reranked finalists


,reranked_rank,baseline_rank,rank_shift,parent_asin,title,price,average_rating,final_score
0,1,2,1,B09LFLPSHM,"Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men",70.40,4.7,0.882968
1,2,1,-1,B0C5S9WGT7,SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown),29.31,4.7,0.835888
2,3,3,0,B01NCQ5EPW,"Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Long Lasting",19.99,4.5,0.818208


Supporting evidence


,parent_asin,evidence_source,matched_topic,matched_terms,relevance_score,evidence_text
0,B0C5S9WGT7,metadata,use_case_fit,"[gift, gifting, for, use, price]",-8.722461,Title: SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown) Desc...
1,B0C5S9WGT7,review,use_case_fit,[affordable],-2.409353,"Review title: Exactly what I wanted!. Review: Totally worth every penny, plus it was very affordable. Perfect!"
2,B09LFLPSHM,metadata,use_case_fit,"[for, gifting, gift, price]",-7.228302,"Title: Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men Features: LUXURIOUS DE..."
3,B09LFLPSHM,review,use_case_fit,"[for, expensive]",-4.707969,Review title: Five Stars. Review: Perfect tall glasses for the table. Makes your wine taste expensive. (a mind trick)
4,B01NCQ5EPW,metadata,use_case_fit,"[gift, for, gifting, experience, price]",-7.180706,"Title: Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Lo..."
5,B01NCQ5EPW,review,use_case_fit,[good],-3.350008,Review title: good. Review: I like it ،my favorite is oud ya aini 😍😍


Final response object
{
  "original_query": "Find me something good for gifting, not too expensive.",
  "parsed_request": {
    "original_query": "Find me something good for gifting, not too expensive.",
    "user_intent": "gift_search",
    "domain_hint": null,
    "hard_constraints": {
      "max_price": null,
      "min_rating": null,
      "must_include_terms": [],
      "use_case": "gifting",
      "cheaper_than_reference": false,
      "same_source_as_reference": false
    },
    "soft_preferences": [
      "budget_friendly",
      "giftable"
    ],
    "reference_items": [],
    "excluded_brands": [],
    "excluded_terms": [],
    "grounding_needs": [
      "use_case_fit",
      "giftability"
    ],
    "clarification_needed": true,
    "clarification_questions": [
      "What kind of gift or product category do you have in mind?"
    ]
  },
  "retrieved_candidates": [
    {
      "parent_asin": "B0C5S9WGT7",
      "source": "home_and_kitchen",
      "title": "SereneLife Bamboo 

## Result Inspection

The pipeline now feels coherent in the sense that every recommendation can be traced back to explicit parsed signals, visible ranking behavior, and retrieved evidence. It still remains limited in predictable ways: gift requests stay ambiguous, evidence can be sparse, and lexical retrieval plus rule-based parsing still leave some nuance on the table.


## Failure Cases And Caveats

- Ambiguous requests still need clarification rather than overconfident recommendations.
- Sparse reviews can leave the system with only metadata-level support.
- Lexical retrieval remains a shallow approximation of user intent.
- The reranker is deliberately transparent rather than fully optimized.

These caveats are part of the system boundary the project is trying to preserve.
